In [1]:
import pandas as pd
import os
import numpy as np
import re
import pyarrow as pa
import pyarrow.parquet as pq

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Key in dataset title

In [4]:
# Load reference dataset-title lookup
ref = pd.read_excel('Datasets/datasets_GBIF_OBIS.xlsx', usecols=['alternateIdentifier', 'title'])

# Build a unique lookup table
ref = ref.dropna(subset=['alternateIdentifier']).drop_duplicates(subset=['alternateIdentifier'], keep='first')
title_lookup = ref.set_index('alternateIdentifier')['title']

# Fill missing/empty datasetTitle values from the lookup
df = pd.read_parquet('MoF/merged_all_mof.parquet')
title_series = df['datasetTitle']
missing_title_mask = title_series.isna() | title_series.astype('string').str.strip().eq('')
mapped_titles = df.loc[missing_title_mask, 'datasetKey'].map(title_lookup)

filled_count = mapped_titles.notna().sum()
df.loc[missing_title_mask, 'datasetTitle'] = mapped_titles

# Save back to the same file
output_path = 'MoF/merged_all_mof.parquet'
df.to_parquet(output_path, index=False)

# Coverage summary
overall_coverage = (df['datasetTitle'].notna().sum() / len(df) * 100).round(2)

source_summary = (
    df.assign(_has_datasetTitle=df['datasetTitle'].notna())
      .groupby('sourceDatabase', dropna=False)
      .agg(
          total_records=('_has_datasetTitle', 'size'),
          datasetTitle_non_null=('_has_datasetTitle', 'sum')
      )
)
source_summary['datasetTitle_coverage_percent'] = (
    source_summary['datasetTitle_non_null'] / source_summary['total_records'] * 100
).round(2)

print(f"Filled datasetTitle for {filled_count:,} rows")
print(f"Overall datasetTitle coverage: {overall_coverage:.2f}%")
print("\nCoverage by sourceDatabase:")
print(source_summary.reset_index().to_string(index=False))
print(f"\nSaved to: {output_path}")

Filled datasetTitle for 2,573,327 rows
Overall datasetTitle coverage: 93.47%

Coverage by sourceDatabase:
sourceDatabase  total_records  datasetTitle_non_null  datasetTitle_coverage_percent
   CRITTERBASE         240561                 240161                          99.83
          GBIF        2500130                2241015                          89.64
          OBIS        1339188                1339188                         100.00
          OGSL           7236                      0                           0.00

Saved to: MoF/merged_all_mof.parquet


# Check column coverage

In [5]:
# Load the parquet file
df = pd.read_parquet('MoF/merged_all_mof.parquet')

# Compute column coverage and dtypes overview
total_rows = len(df)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
})
summary['coverage_percent'] = (summary['non_null_count'] / total_rows * 100).round(2)

# Save to Deduplicate folder
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'column_coverage.csv')
summary.to_csv(output_path, index=False)

# Optional: display the overview
print(summary)
print(f'\nSaved to: {output_path}')

                    column    dtype  non_null_count  coverage_percent
0   acceptedScientificName   string         4087096            100.00
1         acceptedTaxonKey    Int64         2500130             61.17
2                  aphiaID    Int64         4087115            100.00
3          acceptedAphiaID    Int64         4087115            100.00
4              areaSampled  float64          400256              9.79
..                     ...      ...             ...               ...
62               waterBody   string         3563533             87.19
63                    year    Int64         3950203             96.65
64                    Keep      str         3580203             87.60
65                isMarine  boolean         4085616             99.96
66              isBrackish  boolean         3290989             80.52

[67 rows x 4 columns]

Saved to: Deduplicate\column_coverage.csv


# Change 0 in individualCount column to NaN

In [6]:
path = 'MoF/merged_all_mof.parquet'
df = pd.read_parquet(path)

if 'individualCount' in df.columns:
    # ensure numeric float dtype, replace exact zeros with NaN
    df['individualCount'] = pd.to_numeric(df['individualCount'], errors='coerce').astype('float64')
    zero_mask = df['individualCount'] == 0
    replaced = int(zero_mask.sum())
    df.loc[zero_mask, 'individualCount'] = np.nan

    df.to_parquet(path, index=False)
    print(f"Replaced {replaced} zero entries with NaN in: {path}")
else:
    print("Column 'individualCount' not found.")

Replaced 9926 zero entries with NaN in: MoF/merged_all_mof.parquet


# Change unknown gear type to NaN

In [7]:
# Replace 'unknown' / 'Unknown' in gear with missing values and save back to the same file
df = pd.read_parquet(path)

if 'gear' in df.columns:
    gear_mask = df['gear'].isin(['unknown', 'Unknown'])
    replaced = int(gear_mask.sum())
    df.loc[gear_mask, 'gear'] = pd.NA
    df.to_parquet(path, index=False)
    print(f"Replaced {replaced} gear entries with missing values and saved to: {path}")
else:
    print("Column 'gear' not found.")

Replaced 1530 gear entries with missing values and saved to: MoF/merged_all_mof.parquet


# Deduplicate

## Based on datasetTitle

### Identify and inspect duplicates

In [8]:
# Flag datasetTitles that occur in multiple sourceDatabases and report one row per sourceDatabase.
# Preferred sourceDatabase is chosen primarily by record count, then by coverage in the priority columns.
path = 'MoF/merged_all_mof.parquet'
out_xlsx = os.path.join('Deduplicate', 'dataset_deduplication.xlsx')
os.makedirs('Deduplicate', exist_ok=True)

# columns of interest (priority order)
priority_cols = ['areaSampled', 'volumeSampled_m3', 'trawlingDistance_m', 'biomass', 'gear', 'depthA_m', 'individualCount']
coverage_cols = [f"{c}_coverage" for c in priority_cols]

# read only needed columns for performance
cols_needed = ['datasetTitle', 'sourceDatabase', 'decimalLatitudeStart', 'decimalLongitudeStart'] + priority_cols
df = pd.read_parquet(path, columns=cols_needed)

# normalize datasetTitle and drop missing titles
df['datasetTitle'] = df['datasetTitle'].astype('string').str.strip()
df = df.dropna(subset=['datasetTitle'])

def pct_non_null(s):
    return (s.notna().sum() / s.size) * 100

# summarize each datasetTitle/sourceDatabase combination
source_summary = (
    df.groupby(['datasetTitle', 'sourceDatabase'], sort=False)
      .agg(
          n_records=('sourceDatabase', 'size'),
          n_unique_coord_pairs=('decimalLatitudeStart', lambda s: 0),
      )
      .reset_index()
)

# compute unique coordinate-pair counts per datasetTitle/sourceDatabase
source_summary['n_unique_coord_pairs'] = (
    df.groupby(['datasetTitle', 'sourceDatabase'], sort=False)
      .apply(lambda g: g[['decimalLatitudeStart', 'decimalLongitudeStart']].dropna().drop_duplicates().shape[0])
      .values
)

# coverage columns per datasetTitle/sourceDatabase
coverage_summary = (
    df.groupby(['datasetTitle', 'sourceDatabase'], sort=False)[priority_cols]
      .agg(pct_non_null)
      .rename(columns={c: f"{c}_coverage" for c in priority_cols})
      .round(2)
      .reset_index()
)

summary = source_summary.merge(coverage_summary, on=['datasetTitle', 'sourceDatabase'], how='left')

# overall title-level counts used for context
n_sources = df.groupby('datasetTitle', sort=False)['sourceDatabase'].nunique().rename('n_unique_sourceDatabases')
summary = summary.merge(n_sources, on='datasetTitle', how='left')

# pick preferred sourceDatabase per datasetTitle: highest record count, then highest coverage
sort_cols = ['n_records'] + coverage_cols + ['sourceDatabase']
ascending = [False] * (len(sort_cols) - 1) + [True]
preferred_rows = (
    summary.sort_values(by=sort_cols, ascending=ascending)
           .groupby('datasetTitle', sort=False)
           .head(1)[['datasetTitle', 'sourceDatabase']]
           .rename(columns={'sourceDatabase': 'preferred_sourceDatabase'})
)

# mark rows as preferred / not preferred
summary = summary.merge(preferred_rows, on='datasetTitle', how='left')
summary['sourceDatabase_preference'] = np.where(
    summary['sourceDatabase'].eq(summary['preferred_sourceDatabase']),
    'preferred',
    'not_preferred'
)

# keep only datasetTitles that occur in multiple sourceDatabases (flagging duplicates)
conflicts = summary[summary['n_unique_sourceDatabases'] > 1].copy()

# reorder columns for readability
cols_order = [
    'datasetTitle', 'sourceDatabase', 'sourceDatabase_preference', 'preferred_sourceDatabase',
    'n_records', 'n_unique_coord_pairs', 'n_unique_sourceDatabases'
] + coverage_cols
cols_order = [c for c in cols_order if c in conflicts.columns]
conflicts = conflicts[cols_order].sort_values(['datasetTitle', 'sourceDatabase_preference', 'n_records'], ascending=[True, True, False])

# write to Excel
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    conflicts.to_excel(w, sheet_name='conflicting_datasetTitles', index=False)
    # also save the full long-format summary for reference
    summary.sort_values(['datasetTitle', 'sourceDatabase'], ascending=[True, True]).to_excel(
        w,
        sheet_name='all_datasetTitles_summary',
        index=False,
    )

print(f"Saved dataset deduplication report to: {out_xlsx}")

Saved dataset deduplication report to: Deduplicate\dataset_deduplication.xlsx


In [9]:
mof_path = 'MoF/merged_all_mof.parquet'
manual_xlsx = os.path.join('Deduplicate', 'dataset_deduplication_manual.xlsx')
output_path = os.path.join('Deduplicate', 'merged_all_deduplicated.parquet')

# Read the manually edited report
try:
    manual = pd.read_excel(manual_xlsx, sheet_name='all_datasetTitles_summary')
except Exception:
    manual = pd.read_excel(manual_xlsx)

manual_cols = set(manual.columns)

if {'datasetTitle', 'preferred_sourceDatabase'}.issubset(manual_cols):
    title_pref = (
        manual[['datasetTitle', 'preferred_sourceDatabase']]
        .dropna(subset=['datasetTitle'])
        .assign(
            datasetTitle=lambda x: x['datasetTitle'].astype('string').str.strip(),
            preferred_sourceDatabase=lambda x: x['preferred_sourceDatabase'].astype('string').str.strip()
        )
        .drop_duplicates(subset=['datasetTitle'], keep='first')
    )
elif {'datasetTitle', 'sourceDatabase', 'sourceDatabase_preference'}.issubset(manual_cols):
    title_pref = (
        manual.loc[manual['sourceDatabase_preference'].astype('string').str.strip().eq('preferred'),
                   ['datasetTitle', 'sourceDatabase']]
        .dropna(subset=['datasetTitle'])
        .assign(
            datasetTitle=lambda x: x['datasetTitle'].astype('string').str.strip(),
            preferred_sourceDatabase=lambda x: x['sourceDatabase'].astype('string').str.strip()
        )
        [['datasetTitle', 'preferred_sourceDatabase']]
        .drop_duplicates(subset=['datasetTitle'], keep='first')
    )
else:
    raise ValueError(
        "Manual workbook must contain either "
        "['datasetTitle', 'preferred_sourceDatabase'] or "
        "['datasetTitle', 'sourceDatabase', 'sourceDatabase_preference']."
    )

# Load MoF and remove rows not matching the preferred sourceDatabase per datasetTitle
mof = pd.read_parquet(mof_path).copy()
mof['datasetTitle_norm'] = mof['datasetTitle'].astype('string').str.strip()

title_pref = title_pref.rename(columns={'datasetTitle': 'datasetTitle_norm'})
mof = mof.merge(title_pref, on='datasetTitle_norm', how='left')

source_norm = mof['sourceDatabase'].astype('string').str.strip()
pref_norm = mof['preferred_sourceDatabase'].astype('string').str.strip()

remove_mask = pref_norm.notna() & source_norm.ne(pref_norm)
removed_count = int(remove_mask.sum())

removed_by_source = (
    mof.loc[remove_mask]
      .groupby('sourceDatabase', dropna=False)
      .size()
      .sort_values(ascending=False)
)

cleaned = mof.loc[~remove_mask].drop(columns=['datasetTitle_norm', 'preferred_sourceDatabase'])
cleaned.to_parquet(output_path, index=False)

print(f"Removed {removed_count:,} records in total")
print("Removed records per sourceDatabase:")
print(removed_by_source.to_string())
print(f"Remaining records: {len(cleaned):,}")
print(f"Saved to: {output_path}")

Removed 170,964 records in total
Removed records per sourceDatabase:
sourceDatabase
OBIS           89719
GBIF           50157
CRITTERBASE    31088
Remaining records: 3,916,151
Saved to: Deduplicate\merged_all_deduplicated.parquet


## Based on taxonList

### Identify duplicates

In [10]:
# Create stations with rounded coords, taxon lists, coverage cols and duplicate flags, then save

# Read only needed columns for performance
cols_needed = [
    'decimalLatitudeStart', 'decimalLongitudeStart',
    'year', 'startDayOfYear', 'datasetTitle',
    'acceptedAphiaID',
    'areaSampled', 'volumeSampled_m3', 'trawlingDistance_m',
    'biomass', 'gear', 'depthA_m', 'individualCount'
 ]
df = pd.read_parquet('Deduplicate/merged_all_deduplicated.parquet', columns=cols_needed)

# 1) rounded coordinate columns (4 decimals)
df['lat_round4'] = df['decimalLatitudeStart'].round(4)
df['lon_round4'] = df['decimalLongitudeStart'].round(4)

# Prepare grouping keys: use sentinels so missing values form their own groups but don't break groupby
YEAR_SENTINEL = -999999
DAY_SENTINEL = -999999
TITLE_SENTINEL = '<MISSING>'

df['year_grp'] = df['year'].fillna(YEAR_SENTINEL).astype('Int64')
df['startDay_grp'] = df['startDayOfYear'].fillna(DAY_SENTINEL).astype('Int64')
# datasetTitle may be string; fillna with sentinel and cast to string
df['datasetTitle_grp'] = df['datasetTitle'].fillna(TITLE_SENTINEL).astype(str)

group_cols = ['lat_round4', 'lon_round4', 'year_grp', 'startDay_grp', 'datasetTitle_grp']

# coverage source cols and their output names
source_cols = ['areaSampled', 'volumeSampled_m3', 'trawlingDistance_m', 'biomass', 'gear', 'depthA_m', 'individualCount']
coverage_names = [f"{c}_coverage" for c in source_cols]

def pct_non_null(s):
    return (s.notna().sum() / s.size) * 100

# Aggregation
agg_dict = {
    'acceptedAphiaID': lambda s: sorted({int(x) for x in pd.unique(s.dropna())}),
}
for col, covname in zip(source_cols, coverage_names):
    agg_dict[col] = pct_non_null

# Perform groupby aggregation
grouped = df.groupby(group_cols, dropna=False)
stations = grouped.agg(agg_dict)

# grouped.agg with mixed dict can produce MultiIndex columns; normalize:
if isinstance(stations.columns, pd.MultiIndex):
    stations.columns = [
        'taxonList' if c[0] == 'acceptedAphiaID' else (f"{c[0]}_coverage" if c[0] in source_cols else c[0])
        for c in stations.columns.to_flat_index()
    ]

# Ensure we have taxonList and coverage columns
if 'taxonList' not in stations.columns:
    if 'acceptedAphiaID' in stations.columns:
        stations = stations.rename(columns={'acceptedAphiaID': 'taxonList'})
    else:
        stations['taxonList'] = grouped['acceptedAphiaID'].agg(lambda s: sorted({int(x) for x in pd.unique(s.dropna())}))

# compute recordCount explicitly
stations['recordCount'] = grouped.size().astype(int)

# For coverage columns ensure names and rounding
for col, covname in zip(source_cols, coverage_names):
    if covname not in stations.columns:
        if col in stations.columns:
            stations = stations.rename(columns={col: covname})
        else:
            stations[covname] = grouped[col].agg(pct_non_null)
    stations[covname] = stations[covname].round(2)

# Reset index to columns
stations = stations.reset_index()

# Restore original missing indicators to NA for output year/startDay/datasetTitle
stations['year'] = stations['year_grp'].replace({YEAR_SENTINEL: pd.NA}).astype('Int64')
stations['startDayOfYear'] = stations['startDay_grp'].replace({DAY_SENTINEL: pd.NA}).astype('Int64')
stations['datasetTitle'] = stations['datasetTitle_grp'].replace({TITLE_SENTINEL: pd.NA})
# drop group helper cols if desired (keep lat_round4/lon_round4)
stations = stations.drop(columns=['year_grp', 'startDay_grp', 'datasetTitle_grp'])

# station id
stations = stations.reset_index(drop=True)
stations['station_id'] = stations.index.astype(int)

# Add taxonList_key as tuple (useful later and consistent with notebook variables)
stations['taxonList_key'] = stations['taxonList'].apply(lambda x: tuple(x))

# 4) flag duplicates based on rounded coords + year + startDayOfYear
# 4) flag duplicates based on rounded coords + year + startDayOfYear + taxonList
stations['_year_dup'] = stations['year'].fillna(YEAR_SENTINEL)
stations['_day_dup'] = stations['startDayOfYear'].fillna(DAY_SENTINEL)

dup_keys = ['lat_round4', 'lon_round4', '_year_dup', '_day_dup', 'taxonList_key']

# Priority order for keeping (highest priority first)
priority_cov_cols = [
    'areaSampled_coverage', 'volumeSampled_m3_coverage', 'trawlingDistance_m_coverage',
    'biomass_coverage', 'gear_coverage', 'depthA_m_coverage', 'individualCount_coverage'
 ]
# tie-breaker: recordCount
priority_cols_extended = priority_cov_cols + ['recordCount']

# Mark duplicates: keep best-ranked station per duplicate-group (best = max in lexicographic priority order)
stations['duplicate'] = True
keepers = (
    stations
    .sort_values(by=priority_cols_extended, ascending=False)
    .groupby(dup_keys, sort=False)
    .head(1)
    .index
)
stations.loc[keepers, 'duplicate'] = False

# Cleanup helper cols
stations = stations.drop(columns=['_year_dup', '_day_dup'])

# Reorder/select columns for output (keep requested and helpful metadata)
out_cols = [
    'decimalLatitudeStart', 'decimalLongitudeStart', 'lat_round4', 'lon_round4',
    'year', 'startDayOfYear', 'datasetTitle',
    'recordCount', 'taxonList', 'taxonList_key'
 ] + priority_cov_cols + ['duplicate', 'station_id']
final_cols = [c for c in out_cols if c in stations.columns]
stations_output = stations[final_cols].copy()

# Save to Deduplicate/stations.parquet
os.makedirs('Deduplicate', exist_ok=True)
output_path = os.path.join('Deduplicate', 'stations.parquet')
stations_output.to_parquet(output_path, index=False)
print(f"Saved stations to: {output_path}")

Saved stations to: Deduplicate\stations.parquet


### Inspect duplicates

In [11]:
# Export suspected duplicate groups for easy inspection
# Reads Deduplicate/stations.parquet (already created earlier) and writes:
# - Deduplicate/suspected_duplicates.xlsx : all rows belonging to groups with >1 station
# - Deduplicate/suspected_duplicates_summary.xlsx : one-row summary per group with station ids and flags

in_path = os.path.join('Deduplicate', 'stations.parquet')
out_expanded = os.path.join('Deduplicate', 'suspected_duplicates.xlsx')
out_summary = os.path.join('Deduplicate', 'suspected_duplicates_summary.xlsx')

df_stations = pd.read_parquet(in_path)

def normalize_taxon_key(value):
    if isinstance(value, tuple):
        return value
    if isinstance(value, np.ndarray):
        return tuple(value.tolist())
    if isinstance(value, list):
        return tuple(value)
    if pd.isna(value):
        return tuple()
    return (value,)

df_stations['taxonList_key'] = df_stations['taxonList_key'].apply(normalize_taxon_key)

group_cols = ['lat_round4', 'lon_round4', 'year', 'startDayOfYear', 'taxonList_key']

# group size per rounded-location+date
df_stations['group_size'] = df_stations.groupby(group_cols)['station_id'].transform('size')

# keep only groups with more than one station (suspected duplicates)
suspected = df_stations[df_stations['group_size'] > 1].copy()

# sort to show kept (duplicate==False) first within each group
suspected = suspected.sort_values(group_cols + ['duplicate', 'station_id'], ascending=[True, True, True, True, True, True, True])

# write expanded table for row-by-row inspection
suspected.to_excel(out_expanded, index=False)

# write a compact summary per group
summary = (
    suspected
    .groupby(group_cols, dropna=False)
    .agg(
        group_size=('station_id', 'size'),
        station_ids=('station_id', lambda s: ','.join(map(str, s.tolist()))),
        duplicate_flags=('duplicate', lambda s: ','.join(map(str, s.tolist()))),
        datasetTitles=('datasetTitle', lambda s: '; '.join(pd.unique(s.dropna().astype(str)).tolist())),
        taxonLists=('taxonList', lambda s: '; '.join(pd.unique(s.dropna().astype(str)).tolist()))
    )
    .reset_index()
)
summary.to_excel(out_summary, index=False)

print(f"Expanded file: {out_expanded}")
print(f"Summary file:  {out_summary}")

Expanded file: Deduplicate\suspected_duplicates.xlsx
Summary file:  Deduplicate\suspected_duplicates_summary.xlsx


### Remove duplicates

In [12]:
# Remove records belonging to stations labeled duplicate=True and save the result

mof_path = 'Deduplicate/merged_all_deduplicated.parquet'
stations_path = os.path.join('Deduplicate', 'stations.parquet')
output_path = os.path.join('Deduplicate', 'merged_all_deduplicated.parquet')

df = pd.read_parquet(mof_path)
stations = pd.read_parquet(stations_path)

# Build matching keys
df = df.copy()
df['lat_round4'] = df['decimalLatitudeStart'].round(4)
df['lon_round4'] = df['decimalLongitudeStart'].round(4)
df['year_key'] = df['year'].fillna(YEAR_SENTINEL).astype('Int64')
df['day_key'] = df['startDayOfYear'].fillna(DAY_SENTINEL).astype('Int64')
df['title_key'] = df['datasetTitle'].fillna(TITLE_SENTINEL).astype(str)

stations = stations.copy()
stations['year_key'] = stations['year'].fillna(YEAR_SENTINEL).astype('Int64')
stations['day_key'] = stations['startDayOfYear'].fillna(DAY_SENTINEL).astype('Int64')
stations['title_key'] = stations['datasetTitle'].fillna(TITLE_SENTINEL).astype(str)

key_cols = ['lat_round4', 'lon_round4', 'year_key', 'day_key', 'title_key']

# Keys for stations marked as duplicates
dup_keys = (
    stations.loc[stations['duplicate'].fillna(False), key_cols]
    .drop_duplicates()
)

# Tag original records that belong to duplicate stations
marked = df.merge(dup_keys.assign(_remove=True), on=key_cols, how='left')
remove_mask = marked['_remove'].fillna(False).astype(bool)

removed_count = int(remove_mask.sum())
removed_by_source = (
    df.loc[remove_mask]
    .groupby('sourceDatabase', dropna=False)
    .size()
    .sort_values(ascending=False)
)

cleaned = df.loc[~remove_mask].drop(columns=key_cols)

cleaned.to_parquet(output_path, index=False)

print(f"Removed {removed_count:,} records")
print("Removed records by sourceDatabase:")
print(removed_by_source.to_string())
print(f"Remaining records: {len(cleaned):,}")
print(f"Saved to: {output_path}")

Removed 149,991 records
Removed records by sourceDatabase:
sourceDatabase
OBIS           123964
GBIF            20927
CRITTERBASE      5100
Remaining records: 3,766,160
Saved to: Deduplicate\merged_all_deduplicated.parquet


# Check column coverage

In [13]:
# Load the parquet file
df = pd.read_parquet('Deduplicate/merged_all_deduplicated.parquet')

# Compute column coverage and dtypes overview
total_rows = len(df)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
})
summary['coverage_percent'] = (summary['non_null_count'] / total_rows * 100).round(2)

# Save to Deduplicate folder
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'column_coverage_deduplicated.csv')
summary.to_csv(output_path, index=False)

# Optional: display the overview
print(summary)
print(f'\nSaved to: {output_path}')

                    column    dtype  non_null_count  coverage_percent
0   acceptedScientificName   string         3766147            100.00
1         acceptedTaxonKey    Int64         2429046             64.50
2                  aphiaID    Int64         3766160            100.00
3          acceptedAphiaID    Int64         3766160            100.00
4              areaSampled  float64          342369              9.09
..                     ...      ...             ...               ...
62               waterBody   string         3451841             91.65
63                    year    Int64         3676798             97.63
64                    Keep      str         3296095             87.52
65                isMarine  boolean         3764758             99.96
66              isBrackish  boolean         3032761             80.53

[67 rows x 4 columns]

Saved to: Deduplicate\column_coverage_deduplicated.csv
